In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 🌍 UNIVERSAL POWER FORECASTING - SOLAR / WIND / HYBRID AUTO DETECT
# ─────────────────────────────────────────────────────────────────────────
#  HOW IT WORKS:
#   • Put ONLY your CSV file(s) in this folder, then run.
#   • One Solar CSV      → SOLAR forecast
#   • One Wind CSV       → WIND forecast
#   • Solar + Wind CSVs  → HYBRID forecast (both merged on row index)
# ═══════════════════════════════════════════════════════════════════════════

print("="*70)
print("  🌍 UNIVERSAL POWER FORECASTING  (Solar / Wind / Hybrid)")
print("="*70)

# ============================================================================
# 1. INSTALL & IMPORT
# ============================================================================

import subprocess, sys, os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import lightgbm as lgb
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    import lightgbm as lgb

import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
print("✅ Libraries ready")

# ============================================================================
# 2. HELPER – classify a single dataframe as SOLAR, WIND, or UNKNOWN
# ============================================================================

SOLAR_KEYWORDS   = ['ghi', 'irradiance', 'solar', 'pv', 'tcell', 'module', 'aod', 'dni', 'dhi']
WIND_KEYWORDS    = ['wind', 'wspeed', 'wdir', 'turbulence', 'rotor', 'windspeed', 'winddirection']
CAPACITY_WORDS   = ['capacity', 'rated', 'installed', 'nominal']   # constant metadata → exclude as target
OUTPUT_KEYWORDS  = ['pout', 'output', 'generation', 'active_power', 'p_ac', 'pac',
                    'power_mw', 'power_kw', 'power', 'kw', 'mw']

def classify_df(df):
    """Return 'SOLAR', 'WIND', or 'UNKNOWN' based on column names."""
    joined = ' '.join(df.columns).lower()
    is_solar = any(w in joined for w in SOLAR_KEYWORDS)
    is_wind  = any(w in joined for w in WIND_KEYWORDS)
    if is_solar and is_wind:
        return "HYBRID"
    if is_solar:
        return "SOLAR"
    if is_wind:
        return "WIND"
    return "UNKNOWN"

def find_target(df):
    """
    Return the best power-output column name.
    Priority: specific output keywords first, capacity-like words last.
    Constant columns (std ≈ 0) are never selected.
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # remove constants
    variable_cols = [c for c in numeric_cols if df[c].std() > 1e-6]

    for kw in OUTPUT_KEYWORDS:
        for col in variable_cols:
            col_l = col.lower()
            if kw in col_l and not any(ck in col_l for ck in CAPACITY_WORDS):
                return col

    # last resort: highest-variance variable column
    if variable_cols:
        variances = df[variable_cols].var()
        return variances.idxmax()

    raise ValueError("No suitable target (output power) column found in dataset.")

def drop_non_numeric(df):
    """Drop timestamp / ID / string columns."""
    to_drop = []
    for col in df.columns:
        col_l = col.lower()
        if any(kw in col_l for kw in ['timestamp', 'datetime', 'date', 'time', 'id', 'index']):
            to_drop.append(col)
        elif df[col].dtype == object:
            to_drop.append(col)
    if to_drop:
        df = df.drop(columns=to_drop)
    return df

def drop_constants(df, keep_col=None):
    """Remove constant columns (zero variance), optionally preserving keep_col."""
    constants = [c for c in df.columns if df[c].std() < 1e-6 and c != keep_col]
    if constants:
        print(f"   Dropping constant columns: {constants}")
        df = df.drop(columns=constants)
    return df

# ============================================================================
# 3. FIND ALL CSV FILES IN FOLDER
# ============================================================================

print("\n📁 Scanning folder for CSV files...")

ignore_patterns = ['predictions', 'results', 'output', 'forecast']
all_csvs = [f for f in glob.glob("*.csv")
            if not any(p in f.lower() for p in ignore_patterns)]

if not all_csvs:
    print(f"❌  No CSV file found in: {os.getcwd()}")
    raise FileNotFoundError("Put your CSV file(s) in this folder and run again.")

print(f"   Found {len(all_csvs)} CSV file(s): {all_csvs}")

# ============================================================================
# 4. CLASSIFY EACH CSV
# ============================================================================

solar_files   = []
wind_files    = []
unknown_files = []

for fname in all_csvs:
    tmp = pd.read_csv(fname, nrows=5)          # peek at columns only
    kind = classify_df(tmp)
    print(f"   {fname:45s} → {kind}")
    if kind == "SOLAR":
        solar_files.append(fname)
    elif kind == "WIND":
        wind_files.append(fname)
    elif kind == "HYBRID":
        # A single file that already contains both solar + wind columns
        solar_files.append(fname)
        wind_files.append(fname)
    else:
        unknown_files.append(fname)

# ─── Decide run mode ───────────────────────────────────────────────────────
if solar_files and wind_files and solar_files != wind_files:
    RUN_MODE = "HYBRID"
elif solar_files:
    RUN_MODE = "SOLAR"
elif wind_files:
    RUN_MODE = "WIND"
elif unknown_files:
    RUN_MODE = "UNKNOWN"
    solar_files = unknown_files          # fall back: just use whatever is there
else:
    raise FileNotFoundError("Could not classify any CSV file.")

print(f"\n{'='*70}")
print(f"  ▶  RUN MODE SELECTED: {RUN_MODE}")
print(f"{'='*70}")

# ============================================================================
# 5. LOAD & PREPARE DATA ACCORDING TO MODE
# ============================================================================

def load_clean(fname):
    df = pd.read_csv(fname)
    print(f"\n   Loaded '{fname}': {df.shape[0]} rows × {df.shape[1]} cols")
    df = drop_non_numeric(df)
    return df

if RUN_MODE == "HYBRID":
    # ── Load solar and wind separately, find their targets, then merge ──────
    print("\n📥 Loading SOLAR file...")
    df_solar = load_clean(solar_files[0])
    solar_target = find_target(df_solar)
    print(f"   Solar target: {solar_target}")

    print("\n📥 Loading WIND file...")
    df_wind = load_clean(wind_files[0])
    wind_target = find_target(df_wind)
    print(f"   Wind target:  {wind_target}")

    # Rename targets so they don't collide
    df_solar = df_solar.rename(columns={solar_target: "Pout_Solar_kW"})
    df_wind  = df_wind.rename(columns={wind_target:  "Pout_Wind_kW"})

    # Align on row index (same length assumed; trim to shorter)
    min_len = min(len(df_solar), len(df_wind))
    df_solar = df_solar.iloc[:min_len].reset_index(drop=True)
    df_wind  = df_wind.iloc[:min_len].reset_index(drop=True)

    # Merge side-by-side; drop duplicate columns
    df = pd.concat([df_solar, df_wind], axis=1)
    df = df.loc[:, ~df.columns.duplicated()]

    # Hybrid target = sum of both outputs
    df["Pout_Hybrid_kW"] = df["Pout_Solar_kW"] + df["Pout_Wind_kW"]
    TARGET_COL  = "Pout_Hybrid_kW"
    DATA_TYPE   = "HYBRID"
    SOURCE_FILE = f"{solar_files[0]}  +  {wind_files[0]}"

else:
    # ── Single-source (SOLAR or WIND or UNKNOWN) ────────────────────────────
    fname = (solar_files + wind_files + unknown_files)[0]
    print(f"\n📥 Loading file...")
    df = load_clean(fname)
    TARGET_COL  = find_target(df)
    DATA_TYPE   = RUN_MODE
    SOURCE_FILE = fname

print(f"\n✅ Target column : {TARGET_COL}")
print(f"   Range         : {df[TARGET_COL].min():.2f}  →  {df[TARGET_COL].max():.2f}")

# ============================================================================
# 6. FEATURE ENGINEERING
# ============================================================================

print("\n🔧 Engineering features...")

numeric_df = df.select_dtypes(include=[np.number]).copy()
numeric_df = drop_constants(numeric_df, keep_col=TARGET_COL)

# Lag features on the target (no leakage from future)
for lag in [1, 3, 6]:
    numeric_df[f"lag_{lag}"] = numeric_df[TARGET_COL].shift(lag)

numeric_df = numeric_df.dropna()

y = numeric_df[TARGET_COL]
X = numeric_df.drop(columns=[TARGET_COL])

print(f"   Features ({len(X.columns)}): {list(X.columns)}")
print(f"   Final shape: X={X.shape}, y={y.shape}")

# ============================================================================
# 7. TRAIN / TEST SPLIT  (chronological 80 / 20)
# ============================================================================

print("\n📊 Splitting data (80 % train / 20 % test)...")

split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"   Train : {len(X_train):,} rows")
print(f"   Test  : {len(X_test):,} rows")

scaler         = StandardScaler()
X_train_sc     = scaler.fit_transform(X_train)
X_test_sc      = scaler.transform(X_test)

# ============================================================================
# 8. TRAIN LIGHTGBM MODEL
# ============================================================================

print("\n🚀 Training LightGBM model...")

model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    verbose=-1,
    random_state=42
)

model.fit(
    X_train_sc, y_train,
    eval_set=[(X_test_sc, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(period=-1)]
)
print("✅ Training complete")

# ============================================================================
# 9. EVALUATE
# ============================================================================

print("\n📈 Making predictions...")

y_pred = np.maximum(model.predict(X_test_sc), 0)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mask = y_test > 1
mape = (np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100
        if mask.sum() > 0 else float("nan"))

print("\n" + "="*60)
print(f"  {DATA_TYPE} FORECAST RESULTS")
print("="*60)
print(f"  MAE  : {mae:.2f} kW")
print(f"  RMSE : {rmse:.2f} kW")
if not np.isnan(mape):
    print(f"  MAPE : {mape:.1f} %")
print(f"  R²   : {r2:.4f}")
print("="*60)

# ============================================================================
# 10. PLOTS
# ============================================================================

print("\n📊 Creating plots...")

fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.suptitle(f"{DATA_TYPE} Power Forecast  |  R²={r2:.4f}  MAE={mae:.2f} kW",
             fontsize=13, fontweight='bold')

# — Time-series comparison —
n = min(300, len(y_test))
axes[0].plot(y_test.values[:n],  label="Actual",    color="steelblue", alpha=0.85, lw=1.2)
axes[0].plot(y_pred[:n],         label="Predicted", color="tomato",
             alpha=0.85, lw=1.2, linestyle="--")
axes[0].set_xlabel("Hour index")
axes[0].set_ylabel("Power (kW)")
axes[0].set_title(f"First {n} test hours")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# — Scatter: actual vs predicted —
axes[1].scatter(y_test, y_pred, alpha=0.25, s=6, color="steelblue")
lo = min(float(y_test.min()), float(y_pred.min()))
hi = max(float(y_test.max()), float(y_pred.max()))
axes[1].plot([lo, hi], [lo, hi], "r--", lw=1.5, label="Perfect fit")
axes[1].set_xlabel("Actual (kW)");  axes[1].set_ylabel("Predicted (kW)")
axes[1].set_title("Actual vs Predicted")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("forecast_plot.png", dpi=150)
plt.show()
print("✅ Plot saved: forecast_plot.png")

# ============================================================================
# 11. FEATURE IMPORTANCE
# ============================================================================

print("\n🔑 Feature Importance (top 10):")

importance = (pd.DataFrame({"feature": X.columns,
                             "importance": model.feature_importances_})
              .sort_values("importance", ascending=False))

max_imp = importance["importance"].max()
for _, row in importance.head(10).iterrows():
    bar = "█" * max(1, int(row["importance"] / max_imp * 25))
    print(f"   {row['feature']:30s}: {row['importance']:>6.0f}  {bar}")

# ============================================================================
# 12. SAVE OUTPUTS
# ============================================================================

tag = DATA_TYPE.lower()
joblib.dump(model, f"{tag}_model.pkl")
pd.DataFrame({"actual": y_test.values, "predicted": y_pred,
              "error": y_test.values - y_pred}
             ).to_csv(f"{tag}_results.csv", index=False)

print(f"\n✅ Model saved  : {tag}_model.pkl")
print(f"✅ Results saved: {tag}_results.csv")

# ============================================================================
# 13. FINAL SUMMARY
# ============================================================================

print("\n" + "="*70)
print("  ✅ FORECAST COMPLETE")
print("="*70)
print(f"  Source file(s) : {SOURCE_FILE}")
print(f"  Detected type  : {DATA_TYPE}")
print(f"  Target column  : {TARGET_COL}")
print(f"  R² Score       : {r2:.4f}")
print(f"  MAE            : {mae:.2f} kW")
print(f"  RMSE           : {rmse:.2f} kW")
print(f"\n  Files saved:")
print(f"    - {tag}_model.pkl")
print(f"    - {tag}_results.csv")
print(f"    - forecast_plot.png")
print("\n" + "="*70)
print("  🌍 USAGE GUIDE")
print("  ─────────────────────────────────────────────────────────────────")
print("  Put ONLY your CSV(s) in this folder, then run:")
print("    • 1 Solar CSV          →  SOLAR forecast")
print("    • 1 Wind  CSV          →  WIND  forecast")
print("    • 1 Solar + 1 Wind CSV →  HYBRID forecast (auto-merged)")
print("="*70)

# ═══════════════════════════════════════════════════════════════════════════
# 📖 SHAP ANALYSIS - Model Explainability
# Run this cell AFTER training your model
# ═══════════════════════════════════════════════════════════════════════════

print("="*60)
print("  📖 SHAP ANALYSIS - Understanding Model Predictions")
print("="*60)

# Check if model exists
try:
    model
    print("✅ Model found")
except NameError:
    print("❌ No model found! Train the model first.")
    raise NameError("Please run the training cell first")

# Check if SHAP is available
try:
    import shap
    import matplotlib.pyplot as plt
    print("✅ SHAP library ready")
except ImportError:
    print("❌ SHAP not installed. Installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap
    print("✅ SHAP installed")

# Get test data (if available)
try:
    if 'X_test' in dir() or 'X_test_scaled' in dir():
        if 'X_test_scaled' in dir():
            X_for_shap = X_test_scaled
            print(f"✅ Using scaled test data: {X_for_shap.shape}")
        elif 'X_test' in dir():
            X_for_shap = X_test
            print(f"✅ Using test data: {X_for_shap.shape}")
        else:
            # Create sample from training data
            X_for_shap = X_train_scaled if 'X_train_scaled' in dir() else X_train
            print(f"⚠️ Using training data for SHAP: {X_for_shap.shape}")
    else:
        print("❌ No test data found")
        raise NameError("Please run training first")
except:
    print("❌ No data found for SHAP")
    raise

# ============================================================================
# SHAP ANALYSIS
# ============================================================================

print("\n🔍 Running SHAP analysis...")
print("   (This may take 1-3 minutes)")

try:
    # Use smaller sample for faster SHAP
    n_samples = min(100, len(X_for_shap))
    X_sample = X_for_shap[:n_samples]
    
    # Get feature names
    if 'feature_cols' in dir():
        feature_names = feature_cols
    elif 'X_train' in dir() and hasattr(X_train, 'columns'):
        feature_names = X_train.columns
    else:
        feature_names = [f"Feature_{i}" for i in range(X_sample.shape[1])]
    
    print(f"   Using {n_samples} samples, {len(feature_names)} features")
    
    # Create TreeExplainer
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    # ========================================================================
    # PLOT 1: Bar Chart (Feature Importance)
    # ========================================================================
    
    print("\n📊 Creating SHAP Bar Chart...")
    
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names, 
                     show=False, max_display=15, plot_type="bar")
    plt.title(f'SHAP Feature Importance - {DATA_TYPE.upper() if "DATA_TYPE" in dir() else "Power"}', 
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_bar_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("   ✅ Saved: shap_bar_chart.png")
    
    # ========================================================================
    # PLOT 2: Beeswarm Plot (Detailed)
    # ========================================================================
    
    print("\n📊 Creating SHAP Beeswarm Plot...")
    
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names, 
                     show=False, max_display=15, plot_type="dot")
    plt.title(f'SHAP Beeswarm - {DATA_TYPE.upper() if "DATA_TYPE" in dir() else "Power"}', 
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("   ✅ Saved: shap_beeswarm.png")
    
    # ========================================================================
    # PLOT 3: Waterfall Plot (Single Prediction)
    # ========================================================================
    
    print("\n📊 Creating SHAP Waterfall Plot (first prediction)...")
    
    plt.figure(figsize=(10, 5))
    shap.waterfall_plot(shap.Explanation(values=shap_values[0], 
                                         base_values=explainer.expected_value,
                                         data=X_sample[0],
                                         feature_names=feature_names),
                       show=False, max_display=10)
    plt.title('SHAP Waterfall - Single Prediction Breakdown', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("   ✅ Saved: shap_waterfall.png")
    
    # ========================================================================
    # PRINT TOP FEATURES
    # ========================================================================
    
    print("\n📊 Top 10 Most Important Features:")
    print("-" * 40)
    
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top_indices = np.argsort(mean_abs_shap)[::-1][:10]
    
    for i, idx in enumerate(top_indices, 1):
        feature = feature_names[idx] if idx < len(feature_names) else f"Feature_{idx}"
        print(f"   {i:2d}. {feature:<30} {mean_abs_shap[idx]:.4f}")
    
    print("\n" + "="*60)
    print("  ✅ SHAP ANALYSIS COMPLETE")
    print("="*60)
    print("\n  📁 Saved files:")
    print("     - shap_bar_chart.png (feature importance)")
    print("     - shap_beeswarm.png (detailed distribution)")
    print("     - shap_waterfall.png (single prediction)")
    
except Exception as e:
    print(f"\n⚠️ SHAP Error: {e}")
    print("   This could be because:")
    print("   1. Model doesn't support SHAP (non-tree model)")
    print("   2. Data format issue")
    print("   3. Try using simpler model (LightGBM)")

print("\n" + "="*60)
print("  💡 SHAP shows which features drive predictions")
print("  - Red: High feature value increases prediction")
print("  - Blue: High feature value decreases prediction")
print("="*60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 📖 SHAP ANALYSIS - Model Explainability
# Run this cell AFTER training your model
# ═══════════════════════════════════════════════════════════════════════════

print("="*60)
print("  📖 SHAP ANALYSIS - Understanding Model Predictions")
print("="*60)

# Check if model exists
try:
    model
    print("✅ Model found")
except NameError:
    print("❌ No model found! Train the model first.")
    raise NameError("Please run the training cell first")

# Check if SHAP is available
try:
    import shap
    import matplotlib.pyplot as plt
    print("✅ SHAP library ready")
except ImportError:
    print("❌ SHAP not installed. Installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap
    print("✅ SHAP installed")

# Get test data (if available)
try:
    if 'X_test' in dir() or 'X_test_scaled' in dir():
        if 'X_test_scaled' in dir():
            X_for_shap = X_test_scaled
            print(f"✅ Using scaled test data: {X_for_shap.shape}")
        elif 'X_test' in dir():
            X_for_shap = X_test
            print(f"✅ Using test data: {X_for_shap.shape}")
        else:
            # Create sample from training data
            X_for_shap = X_train_scaled if 'X_train_scaled' in dir() else X_train
            print(f"⚠️ Using training data for SHAP: {X_for_shap.shape}")
    else:
        print("❌ No test data found")
        raise NameError("Please run training first")
except:
    print("❌ No data found for SHAP")
    raise

# ============================================================================
# SHAP ANALYSIS
# ============================================================================

print("\n🔍 Running SHAP analysis...")
print("   (This may take 1-3 minutes)")

try:
    # Use smaller sample for faster SHAP
    n_samples = min(100, len(X_for_shap))
    X_sample = X_for_shap[:n_samples]
    
    # Get feature names
    if 'feature_cols' in dir():
        feature_names = feature_cols
    elif 'X_train' in dir() and hasattr(X_train, 'columns'):
        feature_names = X_train.columns
    else:
        feature_names = [f"Feature_{i}" for i in range(X_sample.shape[1])]
    
    print(f"   Using {n_samples} samples, {len(feature_names)} features")
    
    # Create TreeExplainer
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    # ========================================================================
    # PLOT 1: Bar Chart (Feature Importance)
    # ========================================================================
    
    print("\n📊 Creating SHAP Bar Chart...")
    
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names, 
                     show=False, max_display=15, plot_type="bar")
    plt.title(f'SHAP Feature Importance - {DATA_TYPE.upper() if "DATA_TYPE" in dir() else "Power"}', 
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_bar_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("   ✅ Saved: shap_bar_chart.png")
    
    # ========================================================================
    # PLOT 2: Beeswarm Plot (Detailed)
    # ========================================================================
    
    print("\n📊 Creating SHAP Beeswarm Plot...")
    
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names, 
                     show=False, max_display=15, plot_type="dot")
    plt.title(f'SHAP Beeswarm - {DATA_TYPE.upper() if "DATA_TYPE" in dir() else "Power"}', 
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("   ✅ Saved: shap_beeswarm.png")
    
    # ========================================================================
    # PLOT 3: Waterfall Plot (Single Prediction)
    # ========================================================================
    
    print("\n📊 Creating SHAP Waterfall Plot (first prediction)...")
    
    plt.figure(figsize=(10, 5))
    shap.waterfall_plot(shap.Explanation(values=shap_values[0], 
                                         base_values=explainer.expected_value,
                                         data=X_sample[0],
                                         feature_names=feature_names),
                       show=False, max_display=10)
    plt.title('SHAP Waterfall - Single Prediction Breakdown', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("   ✅ Saved: shap_waterfall.png")
    
    # ========================================================================
    # PRINT TOP FEATURES
    # ========================================================================
    
    print("\n📊 Top 10 Most Important Features:")
    print("-" * 40)
    
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top_indices = np.argsort(mean_abs_shap)[::-1][:10]
    
    for i, idx in enumerate(top_indices, 1):
        feature = feature_names[idx] if idx < len(feature_names) else f"Feature_{idx}"
        print(f"   {i:2d}. {feature:<30} {mean_abs_shap[idx]:.4f}")
    
    print("\n" + "="*60)
    print("  ✅ SHAP ANALYSIS COMPLETE")
    print("="*60)
    print("\n  📁 Saved files:")
    print("     - shap_bar_chart.png (feature importance)")
    print("     - shap_beeswarm.png (detailed distribution)")
    print("     - shap_waterfall.png (single prediction)")
    
except Exception as e:
    print(f"\n⚠️ SHAP Error: {e}")
    print("   This could be because:")
    print("   1. Model doesn't support SHAP (non-tree model)")
    print("   2. Data format issue")
    print("   3. Try using simpler model (LightGBM)")

print("\n" + "="*60)
print("  💡 SHAP shows which features drive predictions")
print("  - Red: High feature value increases prediction")
print("  - Blue: High feature value decreases prediction")
print("="*60)